# Precision & Recall: LLM Content Moderation

This notebook walks through how to evaluate an LLM-based classifier using **precision** and **recall** — two of the most important metrics for content moderation and classification tasks.

## The Dataset
We're working with app store reviews that have been **manually labeled** by humans for three types of harmful content:
- `racism`
- `bullying`  
- `sexual`

Reviews that have been labeled (value is `0` or `1`) form the **golden set** — our ground truth.

## The Goal
We'll build an LLM-based classifier that predicts whether a review contains unwanted sexual content, then measure how well it performs against the human labels.

---

## Key Concepts

**Precision** answers: *"Of everything the model flagged, what fraction was actually harmful?"*
- High precision = few false alarms
- `precision = true_positives / (true_positives + false_positives)`

**Recall** answers: *"Of all the actually harmful content, what fraction did the model catch?"*
- High recall = few misses
- `recall = true_positives / (true_positives + false_negatives)`

There's usually a trade-off: tuning your prompt to catch more bad content (higher recall) tends to also flag more innocent content (lower precision), and vice versa. Understanding this trade-off is the whole point of this exercise.


---
## Step 1: Read the Data

In [ ]:
import pandas as pd

df = pd.read_csv("reviews-marked.csv")

print(f"Total rows: {len(df)}")
print(f"\nColumn names: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head(10)


---
## Step 2: Filter to the Golden Set

The **golden set** is the subset of reviews that have been manually labeled by a human. You can identify them because the `racism`, `bullying`, and `sexual` columns have actual values (`0` or `1`) rather than being empty.

These are the only rows we can use to evaluate our classifier — we need ground truth labels to calculate precision and recall.

> **Note:** A value of `1` means the review *contains* that type of content (e.g., unwanted sexual comments). A value of `0` means a human reviewed it and decided it does *not*. Empty/NaN means the review was never labeled for that category — which is **different** from `0`.

Since we're building a classifier for sexual content, we filter down further to only rows where the `sexual` column was actually labeled.


In [ ]:
# A row is in the golden set if ANY of the three label columns has a value.
# notna() returns True if the value is not NaN (i.e., the cell has been filled in).
golden = df[
    df["racism"].notna() | df["bullying"].notna() | df["sexual"].notna()
].copy()

print(f"Total rows in dataset:  {len(df)}")
print(f"Rows in golden set:     {len(golden)}")
print(f"\nHow many have each label filled in:")
print(f"  racism labeled:   {golden['racism'].notna().sum()}")
print(f"  bullying labeled: {golden['bullying'].notna().sum()}")
print(f"  sexual labeled:   {golden['sexual'].notna().sum()}")

# For our sexual content evaluation, we only want rows where sexual was actually labeled.
# NaN means nobody categorized it — it's NOT the same as 0 (human said "no unwanted sexual comments").
golden_sexual = golden[golden["sexual"].notna()].copy()
golden_sexual["sexual"] = golden_sexual["sexual"].astype(int)

print(f"\nRows with a sexual label (our evaluation set): {len(golden_sexual)}")
print(f"  sexual=1 (contains unwanted sexual comments): {golden_sexual['sexual'].sum()}")
print(f"  sexual=0 (does not):                          {(golden_sexual['sexual'] == 0).sum()}")
print(f"\nSample:")
golden_sexual[["Review", "sexual"]].head(5)


---
## Step 3: Write the LLM Prompt

This is the part you'll experiment with! We're going to ask an LLM to read each review and decide whether it contains unwanted sexual content.

The prompt below is a starting point — it's intentionally simple. After you see the precision/recall results, you can come back and tweak the prompt to try to improve the scores.

**Things to experiment with:**
- How specific is your definition of "sexual content"?
- Do you ask for just a yes/no, or ask the model to reason first?
- Do you give examples?
- Do you set a strict or lenient threshold?

We're using [OpenRouter](https://openrouter.ai/) so you can swap in different models without changing the code structure.


In [ ]:
import os
from openai import OpenAI
from pydantic import BaseModel
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

OPENROUTER_API_KEY = os.environ["OPENROUTER_API_KEY"]
MODEL = "google/gemini-2.5-flash"  # ✏️ Change this to try different models

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

print(f"Using model: {MODEL}")
print(f"API key loaded: {'yes' if OPENROUTER_API_KEY else 'NO - check your .env file'}")


### Structured Outputs

Instead of asking the model to reply with "YES" or "NO" and trying to parse that text, we use **structured outputs** — we give the model a schema and it fills it in like a form.

We define two fields:
- `is_sexual` — a boolean (`True`/`False`)
- `reason` — a short string explaining the decision

This is more reliable than parsing free text, and the `reason` field is useful for understanding *why* the model made each call.


In [ ]:
class ModerationResult(BaseModel):
    is_sexual: bool  # True if the review contains unwanted sexual comments
    reason: str      # Short explanation of why the model made this decision


### The Prompt

This is the part you'll experiment with! Write a system prompt and a user prompt that together tell the model what to look for.

**Things to try:**
- How specific is your definition of "unwanted sexual comments"?
- Do you ask the model to reason before deciding?
- Do you give examples of what counts (or doesn't count)?


In [ ]:
import json

def classify_sexual_content(review_text: str) -> ModerationResult:
    system_prompt = """You are a content moderation assistant.
Your job is to detect whether app store reviews contain unwanted sexual comments.
Respond in JSON with two fields: "is_sexual" (boolean) and "reason" (string)."""

    user_prompt = f"""Does the following app store review contain unwanted sexual comments?

Review: {review_text}"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format={"type": "json_object"},
        max_tokens=500,
        temperature=0,
    )

    data = json.loads(response.choices[0].message.content)
    return ModerationResult(**data)


### Sanity Check

Before running on all 331 reviews, test the function on one review to make sure it works.


In [ ]:
test_review = "Great app for meeting people!"
result = classify_sexual_content(test_review)

print(f"Review:    {test_review}")
print(f"is_sexual: {result.is_sexual}")
print(f"reason:    {result.reason}")


---
## Step 4: Run the Classifier on the Golden Set

Now we run `classify_sexual_content()` on every review in `golden_sexual` and store the results in a new column called `sexual_llm`. You'll see a progress bar as it runs.

> **Note:** We're only running this on rows that were actually labeled for sexual content — that's all we need for evaluation. Running it on all 56k rows would cost money and take a long time, and we don't have ground truth labels for those rows anyway.


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

def _run(idx_and_review):
    idx, review_text = idx_and_review
    for attempt in range(3):  # retry up to 3 times on failure
        try:
            result = classify_sexual_content(str(review_text))
            return idx, result.is_sexual, result.reason
        except Exception:
            if attempt == 2:
                raise

reviews = list(golden_sexual["Review"].items())
sexual_llm, reasons = {}, {}

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(_run, item): item for item in reviews}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Classifying reviews"):
        idx, is_sexual, reason = future.result()
        sexual_llm[idx] = int(is_sexual)
        reasons[idx] = reason

golden_sexual["sexual_llm"] = pd.Series(sexual_llm)
golden_sexual["reason"] = pd.Series(reasons)

print(f"LLM flagged as sexual: {golden_sexual['sexual_llm'].sum()}")
print(f"LLM said not sexual:   {(golden_sexual['sexual_llm'] == 0).sum()}")
golden_sexual[["Review", "sexual", "sexual_llm", "reason"]].head(10)


---
## Step 5: Calculate Precision and Recall

Now we compare the LLM's predictions (`sexual_llm`) against the human labels (`sexual`) to compute precision and recall.

First, let's understand the four possible outcomes for each review:

| | Human says: Sexual | Human says: Not Sexual |
|---|---|---|
| **LLM says: Sexual** | ✅ True Positive (TP) | ❌ False Positive (FP) |
| **LLM says: Not Sexual** | ❌ False Negative (FN) | ✅ True Negative (TN) |

- **True Positives (TP):** Correctly caught harmful content
- **False Positives (FP):** Flagged innocent reviews (false alarms)  
- **False Negatives (FN):** Missed actual harmful content
- **True Negatives (TN):** Correctly passed innocent reviews

Then:
- **Precision** = TP / (TP + FP) — "How accurate are the flags?"
- **Recall** = TP / (TP + FN) — "How much harmful content did we catch?"


In [ ]:
human   = golden_sexual["sexual"]      # ground truth (0 or 1)
llm     = golden_sexual["sexual_llm"]  # LLM predictions (0 or 1)

# Count the four outcome types
tp = int(((human == 1) & (llm == 1)).sum())  # both said yes
fp = int(((human == 0) & (llm == 1)).sum())  # LLM said yes, human said no
fn = int(((human == 1) & (llm == 0)).sum())  # LLM said no, human said yes
tn = int(((human == 0) & (llm == 0)).sum())  # both said no

print("Confusion Matrix:")
print(f"  True Positives  (TP): {tp:>4}  — correctly flagged as unwanted sexual comments")
print(f"  False Positives (FP): {fp:>4}  — incorrectly flagged (false alarms)")
print(f"  False Negatives (FN): {fn:>4}  — missed actual unwanted sexual comments")
print(f"  True Negatives  (TN): {tn:>4}  — correctly passed as clean")

# Precision: of all the things we flagged, how many were actually unwanted sexual comments?
precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0

# Recall: of all the actual unwanted sexual comments, how many did we catch?
recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

# F1: harmonic mean of precision and recall — a single balanced score
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

print(f"\n{'='*40}")
print(f"  Precision: {precision:.2%}")
print(f"  Recall:    {recall:.2%}")
print(f"  F1 Score:  {f1:.2%}")
print(f"{'='*40}")
print(f"\nInterpretation:")
print(f"  - The LLM flagged {tp + fp} reviews as containing unwanted sexual comments")
print(f"  - Of those, {tp} matched the human label ({precision:.0%} precision)")
print(f"  - Out of {tp + fn} reviews humans labeled sexual=1, the LLM caught {tp} ({recall:.0%} recall)")


---
## Dig Deeper: Inspect Errors

It's useful to actually *read* the reviews that the model got wrong. Looking at false positives and false negatives often gives you intuition for how to improve the prompt.


In [ ]:
pd.set_option("display.max_colwidth", 300)

# False Positives: LLM flagged it, but humans said it was fine
false_positives = golden_sexual[(human == 0) & (llm == 1)][["Review", "sexual", "sexual_llm", "reason"]]
print(f"False Positives ({len(false_positives)} reviews the LLM incorrectly flagged):")
false_positives


In [ ]:
# False Negatives: Humans flagged it, but LLM missed it
false_negatives = golden_sexual[(human == 1) & (llm == 0)][["Review", "sexual", "sexual_llm", "reason"]]
print(f"False Negatives ({len(false_negatives)} reviews the LLM missed):")
false_negatives
